For time series forecasting we're leveraging AutoGluon Framework since it provides support for a 
large number of probabilistic algorithms including deep learning and transformer based algorithms.

This notebook contains the code for the following steps:
1. Model selection
2. Hyperparameter tuning 

The model selected in this step is used for Model training and deployment in the next steps.

In [ ]:
import pandas as pd

In [ ]:
!pwd

### Reading data saved from Feature creation

#### Read timeseries and static features data

In [ ]:
## read the file 
df_ts = pd.read_csv('../data/final_aggregated_data.csv')
df_ts.rename(columns={'transaction_date':'timestamp','sales_quantity_sum':'target'},inplace = True)
df_ts

In [ ]:
df_static = pd.read_csv('../data/article_metadata.csv')
df_static

In [ ]:
#### some pre processing
static_feature_cols = ['item_height','item_width','item_length','item_weight','article_category']
for col in static_feature_cols:
    if col in df_ts.columns:
        df_ts = df_ts.drop(col, axis=1)

df_ts['timestamp'] = pd.to_datetime(df_ts['timestamp'])
bool_cols = df_ts.select_dtypes(include=['bool']).columns
for col in bool_cols:
    df_ts[col] = df_ts[col].astype(int)
df_ts.dtypes

#### Initialize values and variables

In [ ]:
known_covariates = ['listing_price_mean','net_unit_price','promo_flag','holiday_flag','weekend_flag','holiday_ahead','promo_ahead']
prediction_length = 90
start_date = '2020-01-01'
test_start_date="2023-11-03" 
test_end_date="2024-02-01" 
TARGET = 'target'
trial_name = 'autogluon_ensemble_trial_1'
model_path = f'/home/sagemaker-user/Homepro_AWS_competancy/models/{trial_name}'
TRAIN_TIME_LIMIT =12000

##### perform train-test split 

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
ts = TimeSeriesDataFrame.from_data_frame(df_ts)
ts.static_features = df_static.set_index("item_id")
ts = ts.sort_index()
print("converted to timeseries df")
filtered_ts = ts.slice_by_time(start_time=pd.Timestamp(start_date), end_time=pd.Timestamp(test_end_date))
train_data, test_data = filtered_ts.train_test_split(prediction_length)

##### min history has to be 2*prediction_length+1 days (prediction_length days is the test period)

In [ ]:
min_date = pd.Timestamp(test_end_date) - pd.Timedelta(days=3*prediction_length+1)
filtered_test_data = test_data.slice_by_time(start_time=min_date, end_time=pd.Timestamp(test_end_date))
test_item_counts = filtered_test_data.groupby(level=0).size()
# Sort the counts in ascending order
sorted_counts = test_item_counts.sort_values()
test_filtered_items_to_remove = sorted_counts[sorted_counts < 3*prediction_length+1].index.tolist()
train_data = train_data[~train_data.index.get_level_values(0).isin(test_filtered_items_to_remove)]
test_data = test_data[~test_data.index.get_level_values(0).isin(test_filtered_items_to_remove)]

### Model selection 

In [ ]:
predictor = TimeSeriesPredictor(
        prediction_length = prediction_length,
        path=f'{model_path}',
        target=TARGET,
        eval_metric='MAPE',#'MASE',#'RMSE',
        known_covariates_names=known_covariates,
        verbosity=4,
)
# ## Training AutoGluon Model
predictor.fit(
        train_data,
        presets='best_quality',
        time_limit=TRAIN_TIME_LIMIT,
        random_seed=12345,
        # hyperparameters={
        #     "DeepAR":{#}
        #         "batch_size":64,
        #         "num_batches_per_epoch": 22,
        #         "hidden_dim": 128,
        #         "max_epochs": 100, 
        #         "context_length": 120,
        #         "num_layers": 3,
        #     },
        # },
        # # hyperparameter_tune_kwargs=None
)

#### we can see that DeepAR is performing the best with the HP configuration suggested
	Hyperparameter tune run: DeepAR/T1
		-0.6924      = Validation score (MAPE)
		99.831  s    = Training runtime
		1.666   s    = Training runtime
	Hyperparameter tune run: DeepAR/T2
		-0.6592      = Validation score (MAPE)
		96.092  s    = Training runtime
		1.901   s    = Training runtime
	Hyperparameter tune run: DeepAR/T3
		-0.6643      = Validation score (MAPE)
		114.307 s    = Training runtime
		4.401   s    = Training runtime
	Trained 3 models while tuning DeepAR.
	-0.6592       = Validation score (-MAPE)
	310.33  s     = Total tuning time
	Best hyperparameter configuration: {'num_layers': 2, 'hidden_size': 59}